In [1]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report, f1_score,
                             precision_score, recall_score,
                             average_precision_score,
                             roc_auc_score)
import json

In [2]:
SEED = 4266
DATA_DIR = "../../data/processed"
# DROP_OUT = 0.3
# LEARNING_RATE = 1e-3
# HIDDEN_DIM = 100
INPUT_DIM = 59
OUTPUT_DIM = 5
# LAYER_DIM = 1
EPOCHS = 20
PATIENCE = 4
MIN_DELTA = 1e-4

#hyperparameter search space
HIDDEN_DIM_CHOICES = [32, 64, 100, 128, 256]
LAYER_DIM_MIN = 1
LAYER_DIM_MAX = 2
DROP_OUT_MIN = 0.0
DROP_OUT_MAX = 0.5
DROP_OUT_STEP = 0.1
LEARNING_RATE_MIN = 1e-4
LEARNING_RATE_MAX = 5e-3
WEIGHT_DECAY_MIN = 1e-6
WEIGHT_DECAY_MAX = 1e-3
OPTIMIZER_CHOICES = ["adam", "adamw", "rmsprop"]
OPTIMIZER_MAP = {
    "adam": torch.optim.Adam,
    "adamw": torch.optim.AdamW,
    "rmsprop": torch.optim.RMSprop
}
BATCH_SIZE_CHOICES = [128, 256, 512]

OBJECTIVE_N_TRIALS = 30

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# feats
X_train_raw = np.load(f"{DATA_DIR}/augmented_X_train.npy")
X_val_raw   = np.load(f"{DATA_DIR}/augmented_X_val.npy")
X_test_raw  = np.load(f"{DATA_DIR}/X_test.npy")

# src-des
# pairs_train_raw = np.load(f"{DATA_DIR}/pairs_train.npy", allow_pickle=True)
# pairs_val_raw   = np.load(f"{DATA_DIR}/pairs_val.npy", allow_pickle=True)
# pairs_test  = np.load(f"{DATA_DIR}/pairs_test.npy", allow_pickle=True)

# labels
y_train_raw = np.load(f"{DATA_DIR}/augmented_y_train.npy")
y_val_raw   = np.load(f"{DATA_DIR}/augmented_y_val.npy")
y_test_raw  = np.load(f"{DATA_DIR}/y_test.npy")


In [5]:
X_train = torch.tensor(X_train_raw, dtype=torch.float32)
y_train = torch.tensor(y_train_raw, dtype=torch.float32)
# y_train = torch.tensor(y_train, dtype=torch.long)
X_test = torch.tensor(X_test_raw, dtype=torch.float32)
y_test = torch.tensor(y_test_raw, dtype=torch.float32)
# y_test = torch.tensor(y_test, dtype=torch.long)
X_val = torch.tensor(X_val_raw, dtype=torch.float32)
y_val = torch.tensor(y_val_raw, dtype=torch.float32)
# y_val = torch.tensor(y_val, dtype=torch.long)

y_train_t = torch.tensor(np.argmax(y_train_raw, axis=1), dtype=torch.long)
y_val_t = torch.tensor(np.argmax(y_val_raw, axis=1), dtype=torch.long)
y_test_t = torch.tensor(np.argmax(y_test_raw, axis=1), dtype=torch.long)

In [6]:
print("shape:", y_train.shape)
print("dtype:", y_train.dtype)
print("ndim: ", y_train.ndim)
print("min/max:", y_train.min().item(), y_train.max().item())

shape: torch.Size([165870, 5])
dtype: torch.float32
ndim:  2
min/max: 0.0 1.0


In [7]:
class_counts = torch.bincount(y_train_t, minlength=OUTPUT_DIM).float()
# class_counts = y_train.sum(dim=0).float()
weight = 1.0 / class_counts
weight = weight / weight.sum() * OUTPUT_DIM
print(weight)

tensor([1.2859e+00, 1.0415e+00, 7.0390e-01, 1.9673e+00, 1.3948e-03])


In [8]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim, dropout):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim  = layer_dim
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim,
                            batch_first=True,
                            dropout=dropout if layer_dim > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.attn    = nn.Linear(hidden_dim, 1)
        self.fc      = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        #we reset the hidden state for each batch of sequences so that they are independent of each other. This is important for training on batches of sequences.
        out, _ = self.lstm(x)                       # (B, T, H) — zeros state by default
        # attention pooling over time
        scores  = self.attn(out).squeeze(-1)         # (B, T)
        #weight of each timestep
        weights = torch.softmax(scores, dim=1)       # (B, T)
        #weighted average
        pooled  = (out * weights.unsqueeze(-1)).sum(1)  # (B, H)
        #regularisation
        pooled  = self.dropout(pooled)
        return self.fc(pooled)

In [9]:
def baseline():
    hidden_dim = 128
    layer_dim = 2
    dropout = 0.2
    lr = 1e-4
    weight_decay = 1e-4
    epochs = 30
    seed = SEED
    patience = 5
    optimizer_name = "adam"
    batch_size = 256
    min_delta = 0

    #set seed for reproducibility
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)

    #pair x and y, shuffle the data, and create batches for training
    loader = DataLoader(
        TensorDataset(X_train, y_train_t),
        batch_size=batch_size, shuffle=True, drop_last=False)

    #create LSTM
    model = LSTMModel(INPUT_DIM, hidden_dim, layer_dim, OUTPUT_DIM, dropout).to(device)
    #calculate cross entropy loss with class weight for multiclass
    criterion = nn.CrossEntropyLoss(weight=weight.to(device))

    #dict to map to library optimizers
    OptimCls = OPTIMIZER_MAP[optimizer_name]
    optimizer = OptimCls(model.parameters(), lr=lr, weight_decay=weight_decay)

    #move to gpu
    X_val_dev = X_val.to(device)
    #ensure on cpu
    y_val_np  = y_val_t.cpu().numpy()

    #trackers for metric, state and early stop
    best_ap, best_state, no_improve = -1.0, None, 0

    #history for plotting
    history = {
        "epoch": [],
        "train_loss": [],
        "val_average_precision": [],
        "best_average_precision": [],
        "no_improve": [],
    }

    #run optimising loop
    for epoch in range(epochs):
        print("epoch ", epoch)
        #model training
        model.train()
        #get stats for history
        epoch_loss, n_batches = 0.0, 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            #cap gradient at 1
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            #weight update
            optimizer.step()

            #update stats
            epoch_loss += loss.item()
            n_batches += 1

        #model evaluation
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_dev)
            val_probs = torch.softmax(val_logits, dim=1).cpu().numpy()

        y_val_onehot = np.eye(OUTPUT_DIM)[y_val_np]
        val_average_precision = average_precision_score(
            y_val_onehot, val_probs, average='macro')

        
        history["epoch"].append(epoch)
        history["train_loss"].append(epoch_loss / n_batches)
        history["val_average_precision"].append(val_average_precision)
        history["best_average_precision"].append(max(best_ap, val_average_precision))
        history["no_improve"].append(no_improve)

        #early stop with patience
        if val_average_precision > best_ap + min_delta:
            best_ap = val_average_precision
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_ap, model, history

In [10]:
average_precision, model, history = baseline()

epoch  0
epoch  1
epoch  2
epoch  3
epoch  4
epoch  5
epoch  6
epoch  7
epoch  8
epoch  9
epoch  10
epoch  11
epoch  12
epoch  13
epoch  14
epoch  15
epoch  16
epoch  17
epoch  18


In [17]:
print(average_precision)
print(history)

0.9920289732171648
{'epoch': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18], 'train_loss': [0.9024228030631756, 0.26506051874179165, 0.15126650437517206, 0.12300573765188111, 0.10748777311878425, 0.08365750768468759, 0.07916467637964453, 0.07811706266304863, 0.07312438933850327, 0.07172333701789656, 0.06523542172807437, 0.06984458216094128, 0.060281127299804435, 0.06224098932943889, 0.06803771330165254, 0.06543130061610425, 0.06742905137338155, 0.0709795816427585, 0.06492922943624886], 'val_average_precision': [0.8792619065098684, 0.9437356885172345, 0.9699975055380354, 0.9813395757690271, 0.9835019437892513, 0.9888881310190467, 0.9886245968678828, 0.9902885794891342, 0.9887546236239203, 0.9879413766573599, 0.990568143406516, 0.9846974078556588, 0.9895082458708858, 0.9920289732171648, 0.9905753870981568, 0.9854660546796044, 0.9849893060067417, 0.9884110085986311, 0.9899965132329207], 'best_average_precision': [0.8792619065098684, 0.9437356885172345, 0.96999750553803

In [18]:
print(history["val_average_precision"])

[0.8792619065098684, 0.9437356885172345, 0.9699975055380354, 0.9813395757690271, 0.9835019437892513, 0.9888881310190467, 0.9886245968678828, 0.9902885794891342, 0.9887546236239203, 0.9879413766573599, 0.990568143406516, 0.9846974078556588, 0.9895082458708858, 0.9920289732171648, 0.9905753870981568, 0.9854660546796044, 0.9849893060067417, 0.9884110085986311, 0.9899965132329207]


In [19]:
params = list(model.parameters())
print(params)

[Parameter containing:
tensor([[-0.0278, -0.0215,  0.0020,  ..., -0.0055,  0.0087,  0.0818],
        [ 0.0282, -0.0260,  0.0684,  ..., -0.0146, -0.0044, -0.0162],
        [ 0.0265, -0.0718, -0.0105,  ...,  0.0493, -0.0217, -0.0220],
        ...,
        [ 0.0100, -0.0293,  0.0174,  ...,  0.0915, -0.0243,  0.0401],
        [-0.0262, -0.0707,  0.0144,  ..., -0.0760,  0.0176,  0.0384],
        [-0.0033, -0.0755, -0.0338,  ..., -0.0344,  0.0288,  0.0302]],
       requires_grad=True), Parameter containing:
tensor([[ 0.0021,  0.0002,  0.0265,  ..., -0.0132, -0.0022,  0.0038],
        [ 0.0403,  0.0384,  0.0669,  ..., -0.0184, -0.0285,  0.0259],
        [ 0.0116,  0.0046,  0.0521,  ..., -0.0303, -0.0016,  0.0474],
        ...,
        [ 0.0582, -0.0227,  0.0222,  ..., -0.0226,  0.0185,  0.0038],
        [ 0.0342,  0.0190,  0.0215,  ..., -0.0195, -0.0523,  0.0353],
        [ 0.0549,  0.0178,  0.0490,  ..., -0.0087, -0.0167,  0.0105]],
       requires_grad=True), Parameter containing:
tensor([ 

In [20]:
torch.save(model.state_dict(), "lstm_model.pt")

In [21]:
def compute_metrics(probs, preds, labels, num_classes=5):
    # multiclass need to use 1 hot
    labels_onehot = np.eye(num_classes)[labels]

    return {
        "f1": f1_score(labels, preds, average="macro", zero_division=0),
        "precision": precision_score(labels, preds, average="macro", zero_division=0),
        "recall": recall_score(labels, preds, average="macro", zero_division=0),
        "pr_auc": average_precision_score(labels_onehot, probs, average="macro"),
        "report": classification_report(
            labels,
            preds,
            target_names=["Backdoor", "CommInj", "DoS", "Reconn", "normal"],
            digits=4,
            zero_division=0
        )
    }
def evaluate_lstm(model, dataloader, loss_fn, device, num_classes=5):
    model.eval()
    val_loss = 0
    all_probs, all_preds, all_labels = [], [], []

    with torch.no_grad():
        # No src, dst, or edge_feat for standard sequence data
        for batch_x, y in dataloader:
            batch_x, y = batch_x.to(device), y.to(device)

            # 2. CHANGE: Forward pass only takes the sequence batch
            logits = model(batch_x)
            
            val_loss += loss_fn(logits, y).item()

            # prob and pred for each batch (This part stays exactly the same)
            all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
            all_preds.append(logits.argmax(dim=1).cpu().numpy())
            all_labels.append(y.cpu().numpy())

    # flatten into 1 array (This part stays exactly the same)
    probs = np.concatenate(all_probs)
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)

    # Compute metrics (This part stays exactly the same)
    metrics = compute_metrics(probs, preds, labels, num_classes)
    metrics["loss"] = val_loss / len(dataloader)

    return metrics

In [22]:
test_metrics = evaluate_lstm(model, DataLoader(TensorDataset(X_test, y_test_t), batch_size=128), nn.CrossEntropyLoss(weight=weight.to(device)), device)
print(
    f"Test Loss: {test_metrics['loss']:.4f} | "
    f"F1: {test_metrics['f1']:.4f} | "
    f"Precision: {test_metrics['precision']:.4f} | "
    f"Recall: {test_metrics['recall']:.4f} | "
    f"PR-AUC: {test_metrics['pr_auc']:.4f}"
)

print("\nClassification Report:")
print(test_metrics["report"])

Test Loss: 0.1099 | F1: 0.9151 | Precision: 0.9194 | Recall: 0.9125 | PR-AUC: 0.9381

Classification Report:
              precision    recall  f1-score   support

    Backdoor     0.7222    0.6667    0.6933        39
     CommInj     0.8750    0.9608    0.9159        51
         DoS     1.0000    1.0000    1.0000        91
      Reconn     1.0000    0.9355    0.9667        31
      normal     0.9998    0.9998    0.9998     47152

    accuracy                         0.9994     47364
   macro avg     0.9194    0.9125    0.9151     47364
weighted avg     0.9994    0.9994    0.9994     47364

